# NB04 — Safe negative mining v2

Sao chép nguyên byte anchors/positives từ dataset identity-v1 và chỉ tái sinh negatives. Dùng detector task-specific, expanded-GT exclusion và provenance/integrity. Chạy diagnostics trước.


In [ ]:
# Pinned to the version used by the audited inference notebook.
!pip install -q ultralytics==8.3.221


In [ ]:
import collections
import hashlib
import json
import os
import random
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import ultralytics
from tqdm.auto import tqdm
from ultralytics import YOLO

# ================= CẤU HÌNH ĐƯỜNG DẪN =================
DATASET_ROOT = '/kaggle/input/datasets/tunphmnguynanh/zalo-video/train'
ANN_PATH = os.path.join(DATASET_ROOT, 'annotations', 'annotations.json')
SAMPLES_DIR = os.path.join(DATASET_ROOT, 'samples')

# Dùng đúng detector task-specific của pipeline inference, không dùng yolov8n COCO.
MINING_MODEL_PATH = '/kaggle/input/YOUR_DATASET/yolo_drone_700.pt'
# Giữ nguyên byte anchors/positives đã train identity-v1; chỉ thay negatives.
FROZEN_POSITIVE_DATASET_DIR = (
    '/kaggle/input/datasets/phamnguyenanhtuan/dataset-matching-new-05/dataset_finetune'
)
REUSE_FROZEN_POSITIVES = True
OUTPUT_DIR = '/kaggle/working/dataset_finetune_mining_v2'
DIAGNOSTICS_FILE = '/kaggle/working/mining_v2_diagnostics.json'

# Chạy diagnostics trước. Sau khi kiểm tra bảng, đổi thành True.
RUN_MINING = False
ALLOW_OVERWRITE = False

# Giữ các tham số dataset baseline đã tạo checkpoint identity-v1.
SAMPLING_RATE = 5
MAX_NEG_PER_POS = 2
NUM_BG_PER_FRAME = 2

# Detector candidate mining khớp confidence đầu vào NB06.
CONF_THRESHOLD = 0.05
RANDOM_SEED = 42
FAIL_ON_INFERENCE_ERROR = True

# Negative phải không giao với GT đã nới. Điều này xử lý bbox lớn chứa target nhỏ,
# trường hợp IoU thuần có thể rất thấp dù target nằm trọn trong crop.
GT_EXCLUSION_MARGIN_RATIO = 0.15
GT_EXCLUSION_MARGIN_PX = 4
MIN_CROP_SIZE = 10

# Background crop theo phân bố kích thước target của đúng video.
BACKGROUND_SCALE_RANGE = (0.75, 1.50)
BACKGROUND_FALLBACK_SIZE_RANGE = (50, 200)
BACKGROUND_MAX_ATTEMPTS = 100
BACKGROUND_MAX_MUTUAL_IOU = 0.50

IDENTITY_OVERRIDES = {}


def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path, payload):
    path = str(path)
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    temp = path + '.tmp'
    with open(temp, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    os.replace(temp, path)


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    box2_area = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def intersection_area(box1, box2):
    width = max(0.0, min(box1[2], box2[2]) - max(box1[0], box2[0]))
    height = max(0.0, min(box1[3], box2[3]) - max(box1[1], box2[1]))
    return width * height


def clip_box(box, width, height):
    x1, y1, x2, y2 = map(float, box)
    x1, x2 = min(max(x1, 0.0), width), min(max(x2, 0.0), width)
    y1, y2 = min(max(y1, 0.0), height), min(max(y2, 0.0), height)
    return [x1, y1, x2, y2]


def valid_box(box, min_size=1):
    return box[2] - box[0] >= min_size and box[3] - box[1] >= min_size


def expand_gt_box(box, width, height):
    bw, bh = box[2] - box[0], box[3] - box[1]
    margin_x = max(GT_EXCLUSION_MARGIN_PX, bw * GT_EXCLUSION_MARGIN_RATIO)
    margin_y = max(GT_EXCLUSION_MARGIN_PX, bh * GT_EXCLUSION_MARGIN_RATIO)
    return clip_box(
        [box[0] - margin_x, box[1] - margin_y, box[2] + margin_x, box[3] + margin_y],
        width, height,
    )


def negative_quality(candidate, gt_boxes, width, height):
    candidate = clip_box(candidate, width, height)
    if not valid_box(candidate, MIN_CROP_SIZE):
        return False, 'too_small', {'max_iou': 0.0, 'max_gt_coverage': 0.0}

    max_iou, max_gt_coverage = 0.0, 0.0
    for gt in gt_boxes:
        gt = clip_box(gt, width, height)
        if not valid_box(gt):
            continue
        inter = intersection_area(candidate, gt)
        gt_area = (gt[2] - gt[0]) * (gt[3] - gt[1])
        max_iou = max(max_iou, compute_iou(candidate, gt))
        max_gt_coverage = max(max_gt_coverage, inter / gt_area if gt_area > 0 else 0.0)
        if intersection_area(candidate, expand_gt_box(gt, width, height)) > 0:
            return False, 'expanded_gt_overlap', {
                'max_iou': max_iou,
                'max_gt_coverage': max_gt_coverage,
            }
    return True, 'clean', {'max_iou': max_iou, 'max_gt_coverage': max_gt_coverage}


def save_crop(frame, box, save_dir, prefix):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = map(int, clip_box(box, width, height))
    if x2 - x1 < MIN_CROP_SIZE or y2 - y1 < MIN_CROP_SIZE:
        return None
    crop = frame[y1:y2, x1:x2]
    filename = f'{prefix}_{x1}_{y1}.jpg'
    output_path = os.path.join(save_dir, filename)
    temp_path = output_path + '.tmp.jpg'
    if not cv2.imwrite(temp_path, crop):
        raise IOError(f'cv2.imwrite failed: {output_path}')
    os.replace(temp_path, output_path)
    return {
        'path': os.path.relpath(output_path, OUTPUT_DIR).replace('\\', '/'),
        'box': [x1, y1, x2, y2],
        'width': x2 - x1,
        'height': y2 - y1,
    }


def resolve_identity(video_id):
    if video_id in IDENTITY_OVERRIDES:
        return IDENTITY_OVERRIDES[video_id]
    match = re.fullmatch(r'(.+)_([01])', video_id)
    return match.group(1) if match else video_id


def load_annotations():
    if not os.path.isfile(ANN_PATH):
        raise FileNotFoundError(f'Annotation file not found: {ANN_PATH}')
    with open(ANN_PATH, 'r', encoding='utf-8') as f:
        records = json.load(f)
    if isinstance(records, dict):
        records = [records]

    gt_map = {}
    duplicate_boxes = 0
    invalid_boxes = []
    for record in records:
        video_id = str(record['video_id'])
        frame_map = gt_map.setdefault(video_id, {})
        for interval in record.get('annotations', []):
            for raw in interval.get('bboxes', []):
                frame_idx = int(raw['frame'])
                box = [float(raw['x1']), float(raw['y1']), float(raw['x2']), float(raw['y2'])]
                if frame_idx < 0 or not valid_box(box):
                    invalid_boxes.append({'video_id': video_id, 'frame': frame_idx, 'box': box})
                    continue
                boxes = frame_map.setdefault(frame_idx, [])
                if box in boxes:
                    duplicate_boxes += 1
                else:
                    boxes.append(box)
    if invalid_boxes:
        raise ValueError(f'Invalid GT boxes found, examples: {invalid_boxes[:5]}')
    return gt_map, duplicate_boxes


def video_target_sizes(frame_map):
    sizes = []
    for boxes in frame_map.values():
        for box in boxes:
            width, height = box[2] - box[0], box[3] - box[1]
            if width >= MIN_CROP_SIZE and height >= MIN_CROP_SIZE:
                sizes.append((width, height))
    return sizes


def propose_background_box(frame_width, frame_height, target_sizes, rng):
    if target_sizes:
        base_width, base_height = rng.choice(target_sizes)
        scale = rng.uniform(*BACKGROUND_SCALE_RANGE)
        crop_width = int(round(base_width * scale))
        crop_height = int(round(base_height * scale))
    else:
        low, high = BACKGROUND_FALLBACK_SIZE_RANGE
        crop_width = rng.randint(low, high)
        crop_height = rng.randint(low, high)
    crop_width = min(max(MIN_CROP_SIZE, crop_width), frame_width)
    crop_height = min(max(MIN_CROP_SIZE, crop_height), frame_height)
    if crop_width >= frame_width or crop_height >= frame_height:
        return None
    x1 = rng.randint(0, frame_width - crop_width)
    y1 = rng.randint(0, frame_height - crop_height)
    return [x1, y1, x1 + crop_width, y1 + crop_height]


def random_crop_background(frame, gt_boxes, target_sizes, rng, num_crops=1):
    height, width = frame.shape[:2]
    accepted, rejection_counts = [], collections.Counter()
    for _ in range(BACKGROUND_MAX_ATTEMPTS):
        if len(accepted) >= num_crops:
            break
        candidate = propose_background_box(width, height, target_sizes, rng)
        if candidate is None:
            rejection_counts['frame_too_small'] += 1
            continue
        clean, reason, metrics = negative_quality(candidate, gt_boxes, width, height)
        if not clean:
            rejection_counts[reason] += 1
            continue
        if any(compute_iou(candidate, previous_box) > BACKGROUND_MAX_MUTUAL_IOU
               for previous_box, _ in accepted):
            rejection_counts['duplicate_background'] += 1
            continue
        accepted.append((candidate, metrics))
    if len(accepted) < num_crops:
        rejection_counts['quota_unfilled'] += num_crops - len(accepted)
    return accepted, rejection_counts


def frozen_positive_tree_sha256():
    digest = hashlib.sha256()
    paths = []
    for subdir in ('anchors', 'positives'):
        base = Path(FROZEN_POSITIVE_DATASET_DIR) / subdir
        paths.extend(path for path in base.rglob('*.jpg') if path.is_file())
    for path in sorted(paths, key=lambda item: item.as_posix()):
        relative = path.relative_to(FROZEN_POSITIVE_DATASET_DIR).as_posix()
        digest.update(relative.encode('utf-8'))
        digest.update(b'\0')
        with path.open('rb') as f:
            while True:
                chunk = f.read(1024 * 1024)
                if not chunk:
                    break
                digest.update(chunk)
    return digest.hexdigest(), len(paths)


def inspect_inputs(gt_map):
    if not os.path.isdir(SAMPLES_DIR):
        raise FileNotFoundError(f'SAMPLES_DIR not found: {SAMPLES_DIR}')
    if not os.path.isfile(MINING_MODEL_PATH):
        raise FileNotFoundError(f'MINING_MODEL_PATH not found: {MINING_MODEL_PATH}')
    if REUSE_FROZEN_POSITIVES:
        for subdir in ('anchors', 'positives'):
            required = os.path.join(FROZEN_POSITIVE_DATASET_DIR, subdir)
            if not os.path.isdir(required):
                raise FileNotFoundError(f'Frozen dataset directory missing: {required}')

    rows, errors = [], []
    sample_ids = sorted(
        name for name in os.listdir(SAMPLES_DIR)
        if os.path.isdir(os.path.join(SAMPLES_DIR, name))
    )
    for video_id in sample_ids:
        folder = os.path.join(SAMPLES_DIR, video_id)
        video_path = os.path.join(folder, 'drone_video.mp4')
        anchor_dir = os.path.join(folder, 'object_images')
        anchors = sorted([
            name for name in os.listdir(anchor_dir)
            if name.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]) if os.path.isdir(anchor_dir) else []
        frame_map = gt_map.get(video_id, {})
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else 0
        cap.release()
        if not os.path.isfile(video_path) or frame_count <= 0:
            errors.append(f'{video_id}: unreadable/missing drone_video.mp4')
        out_of_range = sorted(frame for frame in frame_map if frame >= frame_count)
        if out_of_range:
            errors.append(f'{video_id}: GT frames outside video range, examples={out_of_range[:5]}')
        sampled_positive_boxes = sum(
            len(boxes) for frame, boxes in frame_map.items() if frame % SAMPLING_RATE == 0
        )
        frozen_positive_dir = os.path.join(FROZEN_POSITIVE_DATASET_DIR, 'positives', video_id)
        frozen_positive_files = len(list(Path(frozen_positive_dir).glob('*.jpg')))
        frozen_anchor_files = len(list(
            Path(FROZEN_POSITIVE_DATASET_DIR, 'anchors').glob(f'{video_id}_ref_*.jpg')
        ))
        if REUSE_FROZEN_POSITIVES and (frozen_positive_files == 0 or frozen_anchor_files == 0):
            errors.append(f'{video_id}: frozen positives/anchors are empty')
        rows.append({
            'video_id': video_id,
            'identity_id': resolve_identity(video_id),
            'video_frames': frame_count,
            'annotated_frames': len(frame_map),
            'sampled_positive_boxes': sampled_positive_boxes,
            'source_video_anchors': len(anchors),
            'frozen_positive_files': frozen_positive_files,
            'frozen_anchor_files': frozen_anchor_files,
            'positive_count_delta_vs_annotation_plan': (
                frozen_positive_files - sampled_positive_boxes
            ),
        })

    missing_sample_videos = sorted(set(gt_map) - set(sample_ids))
    if missing_sample_videos:
        errors.append(f'Annotated videos missing from samples: {missing_sample_videos}')
    if errors:
        raise ValueError('Input validation failed:\n' + '\n'.join(errors))
    return rows


def experiment_config(model_hash, frozen_tree_hash, frozen_tree_files):
    return {
        'schema_version': 3,
        'experiment': 'negative_mining_v2',
        'dataset_root': os.path.abspath(DATASET_ROOT),
        'mining_model_path': os.path.abspath(MINING_MODEL_PATH),
        'mining_model_sha256': model_hash,
        'reuse_frozen_positives': REUSE_FROZEN_POSITIVES,
        'frozen_positive_dataset_dir': os.path.abspath(FROZEN_POSITIVE_DATASET_DIR),
        'frozen_positive_tree_sha256': frozen_tree_hash,
        'frozen_positive_tree_jpg_files': frozen_tree_files,
        'ultralytics_version': ultralytics.__version__,
        'opencv_version': cv2.__version__,
        'numpy_version': np.__version__,
        'sampling_rate': SAMPLING_RATE,
        'confidence_threshold': CONF_THRESHOLD,
        'max_negative_per_positive': MAX_NEG_PER_POS,
        'background_per_frame': NUM_BG_PER_FRAME,
        'random_seed': RANDOM_SEED,
        'gt_exclusion_margin_ratio': GT_EXCLUSION_MARGIN_RATIO,
        'gt_exclusion_margin_px': GT_EXCLUSION_MARGIN_PX,
        'background_scale_range': list(BACKGROUND_SCALE_RANGE),
        'background_max_mutual_iou': BACKGROUND_MAX_MUTUAL_IOU,
        'filename_provenance': '<video>_f<frame>_(h|bg)<index>_<x1>_<y1>.jpg',
    }


def prepare_output_dir():
    output = Path(OUTPUT_DIR).resolve()
    if output.exists() and any(output.iterdir()):
        if not ALLOW_OVERWRITE:
            raise FileExistsError(
                f'OUTPUT_DIR is not empty: {output}. Use a new directory or set ALLOW_OVERWRITE=True.'
            )
        shutil.rmtree(output)
    for subdir in ('anchors', 'positives', 'negatives/hard', 'negatives/background'):
        (output / subdir).mkdir(parents=True, exist_ok=True)


def stage_frozen_positives():
    if not REUSE_FROZEN_POSITIVES:
        raise RuntimeError('Mining-v2 controlled ablation requires REUSE_FROZEN_POSITIVES=True')
    for subdir in ('anchors', 'positives'):
        shutil.copytree(
            os.path.join(FROZEN_POSITIVE_DATASET_DIR, subdir),
            os.path.join(OUTPUT_DIR, subdir),
            dirs_exist_ok=True,
        )


def copy_anchors(video_id):
    source_dir = os.path.join(SAMPLES_DIR, video_id, 'object_images')
    if not os.path.isdir(source_dir):
        return 0
    image_names = sorted([
        name for name in os.listdir(source_dir)
        if name.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    for index, image_name in enumerate(image_names):
        shutil.copyfile(
            os.path.join(source_dir, image_name),
            os.path.join(OUTPUT_DIR, 'anchors', f'{video_id}_ref_{index}.jpg'),
        )
    return len(image_names)


def validate_mining_output(summary, diagnostic_rows):
    root = Path(OUTPUT_DIR)
    positive_files = sorted((root / 'positives').glob('*/*.jpg'))
    hard_files = sorted((root / 'negatives' / 'hard').glob('*.jpg'))
    background_files = sorted((root / 'negatives' / 'background').glob('*.jpg'))
    anchor_files = sorted((root / 'anchors').glob('*.jpg'))
    expected_positives = sum(row['frozen_positive_files'] for row in diagnostic_rows)
    expected_anchors = sum(row['frozen_anchor_files'] for row in diagnostic_rows)
    expected = {
        'positive_files': expected_positives,
        'hard_files': summary['totals']['hard_negatives'],
        'background_files': summary['totals']['background_negatives'],
        'anchor_files': expected_anchors,
    }
    actual = {
        'positive_files': len(positive_files),
        'hard_files': len(hard_files),
        'background_files': len(background_files),
        'anchor_files': len(anchor_files),
    }
    if actual != expected:
        raise RuntimeError(f'Output file count mismatch: actual={actual}, expected={expected}')

    manifest_path = root / 'negative_manifest.jsonl'
    with manifest_path.open('r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]
    if len(records) != len(hard_files) + len(background_files):
        raise RuntimeError('Negative manifest row count does not match crop files')
    provenance_pattern = re.compile(r'.+_f\d+_(h|bg)\d+_\d+_\d+\.jpg')
    seen_paths = set()
    for record in records:
        relative_path = record['path']
        filename = os.path.basename(relative_path)
        if not provenance_pattern.fullmatch(filename):
            raise RuntimeError(f'Invalid negative provenance filename: {filename}')
        if relative_path in seen_paths:
            raise RuntimeError(f'Duplicate negative path in manifest: {relative_path}')
        seen_paths.add(relative_path)
        if not (root / relative_path).is_file():
            raise RuntimeError(f'Manifest points to missing crop: {relative_path}')
        if record['max_iou'] != 0.0 or record['max_gt_coverage'] != 0.0:
            raise RuntimeError(f'Negative crop overlaps GT: {relative_path}')
    return {
        'status': 'passed',
        'expected_file_counts': expected,
        'actual_file_counts': actual,
        'negative_manifest_rows': len(records),
        'unique_negative_paths': len(seen_paths),
        'all_negative_gt_overlap_metrics_zero': True,
    }


def mine_dataset(gt_map, diagnostic_rows, config):
    prepare_output_dir()
    stage_frozen_positives()
    rng = random.Random(RANDOM_SEED)
    model = YOLO(MINING_MODEL_PATH)
    summary = {
        'status': 'running',
        'config': config,
        'config_sha256': canonical_hash(config),
        'rows': [],
        'totals': collections.Counter(),
        'warnings': [],
    }
    negative_manifest_path = os.path.join(OUTPUT_DIR, 'negative_manifest.jsonl')

    with open(negative_manifest_path, 'w', encoding='utf-8') as manifest_file:
        for diagnostic in tqdm(diagnostic_rows, desc='Mining videos'):
            video_id = diagnostic['video_id']
            video_path = os.path.join(SAMPLES_DIR, video_id, 'drone_video.mp4')
            frame_map = gt_map.get(video_id, {})
            target_sizes = video_target_sizes(frame_map)
            row = dict(diagnostic)
            row.update({
                'positive_crops': diagnostic['frozen_positive_files'],
                'hard_negatives': 0,
                'background_negatives': 0,
                'sampled_frames': 0,
                'inference_errors': 0,
                'detector_candidates': 0,
                'hard_quota_requested': 0,
                'hard_quota_unfilled': 0,
                'rejections': collections.Counter(),
            })
            row['copied_anchors'] = diagnostic['frozen_anchor_files']

            cap = cv2.VideoCapture(video_path)
            frame_idx = 0
            while cap.isOpened():
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_idx % SAMPLING_RATE != 0:
                    frame_idx += 1
                    continue
                row['sampled_frames'] += 1
                current_gt = frame_map.get(frame_idx, [])
                height, width = frame.shape[:2]

                # Positives/anchors were copied byte-for-byte before the video loop.

                hard_limit = max(1, len(current_gt)) * MAX_NEG_PER_POS
                row['hard_quota_requested'] += hard_limit
                try:
                    results = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)
                except Exception as error:
                    row['inference_errors'] += 1
                    if FAIL_ON_INFERENCE_ERROR:
                        cap.release()
                        raise RuntimeError(
                            f'YOLO mining failed: {video_id} frame {frame_idx}'
                        ) from error
                    results = []

                candidates = []
                for result in results:
                    boxes = result.boxes.xyxy.cpu().numpy()
                    confidences = result.boxes.conf.cpu().numpy()
                    candidates.extend(zip(boxes, confidences))
                candidates.sort(key=lambda pair: float(pair[1]), reverse=True)
                row['detector_candidates'] += len(candidates)

                selected_hard = []
                for box, confidence in candidates:
                    clean, reason, metrics = negative_quality(box, current_gt, width, height)
                    if not clean:
                        row['rejections'][reason] += 1
                        continue
                    clipped = clip_box(box, width, height)
                    if any(compute_iou(clipped, prior) > 0.90 for prior in selected_hard):
                        row['rejections']['duplicate_hard'] += 1
                        continue
                    hard_index = len(selected_hard)
                    saved = save_crop(
                        frame, clipped, os.path.join(OUTPUT_DIR, 'negatives/hard'),
                        prefix=f'{video_id}_f{frame_idx}_h{hard_index}',
                    )
                    if saved is None:
                        row['rejections']['too_small'] += 1
                        continue
                    selected_hard.append(clipped)
                    record = {
                        **saved,
                        'video_id': video_id,
                        'identity_id': resolve_identity(video_id),
                        'frame': frame_idx,
                        'kind': 'hard',
                        'detector_confidence': float(confidence),
                        **metrics,
                    }
                    manifest_file.write(json.dumps(record, ensure_ascii=False) + '\n')
                    row['hard_negatives'] += 1
                    if len(selected_hard) >= hard_limit:
                        break
                row['hard_quota_unfilled'] += hard_limit - len(selected_hard)

                background, background_rejections = random_crop_background(
                    frame, current_gt, target_sizes, rng, num_crops=NUM_BG_PER_FRAME
                )
                row['rejections'].update(background_rejections)
                for background_index, (box, metrics) in enumerate(background):
                    saved = save_crop(
                        frame, box, os.path.join(OUTPUT_DIR, 'negatives/background'),
                        prefix=f'{video_id}_f{frame_idx}_bg{background_index}',
                    )
                    if saved is None:
                        row['rejections']['too_small'] += 1
                        continue
                    record = {
                        **saved,
                        'video_id': video_id,
                        'identity_id': resolve_identity(video_id),
                        'frame': frame_idx,
                        'kind': 'background',
                        'detector_confidence': None,
                        **metrics,
                    }
                    manifest_file.write(json.dumps(record, ensure_ascii=False) + '\n')
                    row['background_negatives'] += 1
                frame_idx += 1
            cap.release()

            row['rejections'] = dict(sorted(row['rejections'].items()))
            summary['rows'].append(row)
            for key in (
                'sampled_frames', 'positive_crops', 'hard_negatives', 'background_negatives',
                'inference_errors', 'detector_candidates', 'hard_quota_requested',
                'hard_quota_unfilled',
            ):
                summary['totals'][key] += row[key]
            summary['totals']['anchors'] += row['copied_anchors']
            atomic_write_json(os.path.join(OUTPUT_DIR, 'mining_summary.json'), {
                **summary, 'totals': dict(summary['totals'])
            })

    summary['totals'] = dict(summary['totals'])
    rejection_totals = collections.Counter()
    for row in summary['rows']:
        rejection_totals.update(row['rejections'])
    summary['rejection_totals'] = dict(sorted(rejection_totals.items()))
    summary['totals']['negatives'] = (
        summary['totals']['hard_negatives'] + summary['totals']['background_negatives']
    )
    positives = summary['totals']['positive_crops']
    summary['negative_to_positive_ratio'] = (
        summary['totals']['negatives'] / positives if positives else None
    )
    summary['integrity'] = validate_mining_output(summary, diagnostic_rows)
    summary['status'] = 'completed'
    atomic_write_json(os.path.join(OUTPUT_DIR, 'mining_summary.json'), summary)
    atomic_write_json(os.path.join(OUTPUT_DIR, 'mining_config.json'), config)
    return summary


def print_diagnostics(rows, duplicate_boxes, config):
    print('\nVIDEO / IDENTITY / INPUT SUMMARY')
    for row in rows:
        print(
            f"  {row['video_id']:20s} identity={row['identity_id']:15s} "
            f"frames={row['video_frames']:5d} annotated={row['annotated_frames']:5d} "
            f"planned_pos={row['sampled_positive_boxes']:4d} "
            f"frozen_pos={row['frozen_positive_files']:4d} "
            f"delta={row['positive_count_delta_vs_annotation_plan']:+4d} "
            f"frozen_anchors={row['frozen_anchor_files']}"
        )
    print(f"\nDetector SHA256: {config['mining_model_sha256']}")
    print(f'Annotation duplicate boxes ignored: {duplicate_boxes}')
    print('Negative rule: zero overlap with expanded GT; not IoU-only.')


def main():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    gt_map, duplicate_boxes = load_annotations()
    diagnostic_rows = inspect_inputs(gt_map)
    model_hash = file_sha256(MINING_MODEL_PATH)
    frozen_tree_hash, frozen_tree_files = frozen_positive_tree_sha256()
    config = experiment_config(model_hash, frozen_tree_hash, frozen_tree_files)
    diagnostics = {
        'status': 'diagnostics_only' if not RUN_MINING else 'ready_to_mine',
        'config': config,
        'config_sha256': canonical_hash(config),
        'duplicate_annotation_boxes_ignored': duplicate_boxes,
        'rows': diagnostic_rows,
    }
    atomic_write_json(DIAGNOSTICS_FILE, diagnostics)
    print_diagnostics(diagnostic_rows, duplicate_boxes, config)

    if not RUN_MINING:
        print(f'\nDiagnostics saved: {DIAGNOSTICS_FILE}')
        print('Review mapping/counts, then set RUN_MINING=True with a new OUTPUT_DIR.')
        return diagnostics

    try:
        summary = mine_dataset(gt_map, diagnostic_rows, config)
    except Exception as error:
        if os.path.isdir(OUTPUT_DIR):
            atomic_write_json(os.path.join(OUTPUT_DIR, 'mining_failed.json'), {
                'status': 'failed',
                'config': config,
                'error_type': type(error).__name__,
                'error': str(error),
            })
        raise
    print('\nMINING COMPLETED')
    print(json.dumps(summary['totals'], indent=2))
    print(f"Negative/positive ratio: {summary['negative_to_positive_ratio']:.3f}")
    print(f"Artifacts: {OUTPUT_DIR}")
    return summary


In [ ]:
if __name__ == '__main__':
    main()
